# Assay Summary

In [1]:
import polars as pl

## 1. Loading data

### 1.1 Clean up allele collection

In [2]:
# ALLELE_COLLECTION_INFO = "../../1_allele_collection/3_outputs"
# clin_var_df = pl.read_csv(f"{ALLELE_COLLECTION_INFO}/varchamp_seq_confirm_alleles_clinvar_gnomad_dbnsfp_struc_scores_OMIM_MOI_disease_modules.tsv", 
#                           separator="\t", infer_schema_length=10000
# ).with_columns(
#     pl.col("symbol").str.replace_all(r"[^a-zA-Z0-9]", "").alias("symbol"),
#     pl.col("plddt").cast(pl.Float32).alias("plddt_f32"),
#     pl.col("rsa").cast(pl.Float32).alias("rsa_f32")
# ).with_columns(
#     (
#         pl.when(pl.col("rsa_f32")<.1)
#         .then(pl.lit("<10"))
#         .when(pl.col("rsa_f32")>=.9)
#         .then(pl.lit("$\geqslant$90"))
#         .otherwise(pl.lit("10-90"))
#     ).alias("RSA_Category"),
#     (
#         pl.when(pl.col("plddt_f32")<=50)
#         .then(pl.lit("<=50"))
#         .when((pl.col("plddt_f32")>50)&(pl.col("plddt_f32")<=70))
#         .then(pl.lit("50-70"))
#         .when((pl.col("plddt_f32")>70)&(pl.col("plddt_f32")<=90))
#         .then(pl.lit("70-90"))
#         .otherwise(pl.lit(">90"))
#     ).alias("pLDDT_Category")
# )
# clin_var_df.write_csv("../1_inputs/varchamp_clinvar_gnomad_dbnsfp_disease_modules.tsv", separator="\t")

### 1.2 Merge allele collection with DUAL-IPA, Imaging and PPI assays

In [5]:
allele_collection_inputs = "../1_inputs/"
dualipa_outputs = "../../2_individual_assay_results/dual_ipa/3_outputs"
imaging_outputs = "../../2_individual_assay_results/imaging/3_outputs"
ppi_outputs = "../../2_individual_assay_results/ppi/3_outputs/sqY2H"

allele_col_df = pl.read_csv(f"{allele_collection_inputs}/varchamp_clinvar_gnomad_dbnsfp_disease_modules.tsv", 
                            separator="\t", infer_schema_length=100000)
dualipa = pl.read_csv(f"{dualipa_outputs}/annotated_perturbation_scores.csv", infer_schema_length=100000)
print("DUAL-IPA:", dualipa.shape[0], dualipa.unique(subset="gene_allele").shape[0])
imaging = pl.read_csv(f"{imaging_outputs}/imaging_analyses_summary_clinvar.tsv", separator="\t", infer_schema_length=100000)
print("Imaging:", imaging.shape[0], imaging.unique(subset="gene_allele").shape[0])
ppi = pl.read_csv(f"{ppi_outputs}/6_variants_perturbation_scores_automated_annotated.csv", infer_schema_length=100000)
print("PPI:", ppi.shape[0], ppi.unique(subset="allele_0").shape[0])

DUAL-IPA: 804 804
Imaging: 943 943
PPI: 482 475


In [6]:
al_dual_df = allele_col_df.join(
    dualipa.select([c for c in dualipa.columns if c not in allele_col_df.columns]),
    left_on="gene_variant",
    right_on="gene_allele",
    how="left"
)

al_dual_img = al_dual_df.join(
    imaging.select([c for c in imaging.columns if c not in allele_col_df.columns]),
    left_on="gene_variant",
    right_on="gene_allele",
    how="left",
    suffix="_img"
)

al_dual_img_ppi = al_dual_img.with_columns(
    pl.col("mut_id").cast(pl.Int64).alias("mut_id")
).join(
    ppi.select(["orf_id", "mut_id"]+[c for c in ppi.columns if c not in allele_col_df.columns]),
    left_on=["orf_id", "mut_id"],
    right_on=["orf_id", "mut_id"],
    how="left",
    suffix="_ppi"
)

al_dual_img_ppi

symbol,ensembl_gene_id,orf_id,mut_id,ccsb_mutation_id,ccsb_allele_id,spdi,nt_change,aa_change,collection,entry_plate_orig,entry_well_orig,entry_plate_conso,entry_well_conso,entry_seq_pool,db_plate,db_well,n2h_plate,n2h_well,dualip_plate,dualip_well,mislocalization_plate,mislocalization_well,entry_sequenced,entry_sequence_confirmation_class,db_sequenced,db_sequence_confirmation_class,n2h_sequenced,n2h_sequence_confirmation_class,n2h_ref_sequence_confirmation_class,dualip_sequenced,dualip_sequence_confirmation_class,dualip_ref_sequence_confirmation_class,mislocalization_sequenced,mislocalization_sequence_confirmation_class,mislocalization_ref_sequence_confirmation_class,gene_variant,…,AUROC_BioRep1_GFP,AUROC_BioRep2_Morph,AUROC_BioRep2_AGP,AUROC_BioRep2_Mito,AUROC_BioRep2_DNA,AUROC_BioRep2_GFP,euclid_distance,num_edges,num_zero_wt,num_per_0.0,num_per_0.1,num_per_0.2,num_per_0.3,num_per_0.4,num_per_0.5,num_edge_0.0,num_edge_0.1,num_edge_0.2,num_edge_0.3,num_edge_0.4,num_edge_0.5,frac_per_0.0,frac_per_0.1,frac_per_0.2,frac_per_0.3,frac_per_0.4,frac_per_0.5,normalized_euclid_distance,allele_0,symbol_right_ppi,aa_change_right_ppi,ensembl_gene_id_right,clinvar_clnsig_clean_right,gnomad_af_right,StarStatus_right,ppi_score_final_euclidean,ppi_hit
str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,…,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,f64,f64,f64,bool
"""GBA1""","""ENSG00000177628""",2,6,"""CCSBVarC000001""","""ALE0000584""","""NC_000001.11:155240033:C:G""","""160G>C""","""Val54Leu""","""RC4""","""RC4_Mut_GDEh1026""","""H01""","""GDEhDisVCh_40054""","""F12""",2.0,"""RC4_Mut_GDDh1026""","""H01""",null,null,null,null,null,null,1.0,1.0,1.0,2.0,0.0,null,null,0.0,null,null,0.0,null,null,"""GBA1_Val54Leu""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""GBA1""","""ENSG00000177628""",2,73,"""CCSBVarC000002""","""ALE00000002""","""NC_000001.11:155238225:G:A""","""670C>T""","""Leu224Phe""","""RC4""","""RC4_Mut_GDEh1026""","""E01""","""GDEhDisVCh_40054""","""C12""",2.0,"""RC4_Mut_GDDh1026""","""E01""",null,null,null,null,null,null,1.0,1.0,1.0,1.0,0.0,null,null,0.0,null,null,0.0,null,null,"""GBA1_Leu224Phe""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""GBA1""","""ENSG00000177628""",2,113,"""CCSBVarC000003""","""ALE00000003""","""NC_000001.11:155237453:C:T""","""887G>A""","""Arg296Gln""","""RC4""","""RC4_Mut_GDEh1026""","""F01""","""GDEhDisVCh_40054""","""D12""",2.0,"""RC4_Mut_GDDh1026""","""F01""",null,null,null,null,null,null,1.0,1.0,1.0,7.0,0.0,null,null,0.0,null,null,0.0,null,null,"""GBA1_Arg296Gln""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""GBA1""","""ENSG00000177628""",2,231,"""CCSBVarC000004""","""ALE00000004""","""NC_000001.11:155235252:A:G""","""1448T>C""","""Leu483Pro""","""RC4""","""RC4_Mut_GDEh1026""","""G01""","""GDEhDisVCh_40054""","""E12""",2.0,"""RC4_Mut_GDDh1026""","""G01""",null,null,null,null,null,null,1.0,1.0,1.0,2.0,0.0,null,null,0.0,null,null,0.0,null,null,"""GBA1_Leu483Pro""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""GBA1""","""ENSG00000177628""",2,213510,"""CCSBVarC003869""","""ALE00003869""","""NC_000001.11:155239934:G:A""","""259C>T""","""Arg87Trp""","""CEGS2""","""CegsMutGDEh1035""","""B03""","""GDEhDisVCh_40054""","""B02""",2.0,"""CegsMutGDDh1035""","""B03"

## 2. Analyze assay overlapping

### 2.1 Gene alleles profiled by ALL 3 assays

In [9]:
meta_cols = ["symbol", "gene_variant", "ensembl_gene_id", "spdi", "orf_id", "mut_id", 
             "nt_change", "aa_change", "clinvar_clnsig_clean", "gnomad_af", "StarStatus",
             "AlphaMissense_score", "AlphaMissense_rankscore", "AlphaMissense_pred",
             "ESM1b_score", "ESM1b_rankscore", "ESM1b_pred",
             "plddt_f32", "rsa_f32", "RSA_Category", "pLDDT_Category"]
dualipa_cols = ["dualipa_abun_score", "dualipa_abun_change", "dualipa_abun_chg_dir"]
img_cols = ["AUROC_Mean_GFP", "Altered_95th_perc_both_batches_GFP"] ## "AUROC_Mean_DNA", "Altered_95th_perc_both_batches_DNA"
ppi_cols = ["ppi_score_final_euclidean", "ppi_hit"]

al_dual_img_ppi_key_cols = al_dual_img_ppi.select(
    pl.col(meta_cols + dualipa_cols + img_cols + ppi_cols)
)

In [15]:
al_dual_img_ppi_key_cols.filter(
    (pl.col("dualipa_abun_change")=="Y") & (pl.col("dualipa_abun_chg_dir")=="Increase")
)

symbol,gene_variant,ensembl_gene_id,spdi,orf_id,mut_id,nt_change,aa_change,clinvar_clnsig_clean,gnomad_af,StarStatus,AlphaMissense_score,AlphaMissense_rankscore,AlphaMissense_pred,ESM1b_score,ESM1b_rankscore,ESM1b_pred,plddt_f32,rsa_f32,RSA_Category,pLDDT_Category,dualipa_abun_score,dualipa_abun_change,dualipa_abun_chg_dir,AUROC_Mean_GFP,Altered_95th_perc_both_batches_GFP,ppi_score_final_euclidean,ppi_hit
str,str,str,str,i64,i64,str,str,str,f64,f64,str,str,str,str,str,str,f64,f64,str,str,f64,str,str,f64,bool,f64,bool
"""KLHL3""","""KLHL3_Met78Val""","""ENSG00000146021""","""NC_000005.10:137709759:T:C""",1256,214007,"""232A>G""","""Met78Val""","""1_Pathogenic""",null,null,"""0.947""","""0.86564""","""P""","""-13.846""","""0.92561""","""D""",91.78,0.143617,"""10-90""",""">90""",2.858194,"""Y""","""Increase""",0.751687,false,null,null
"""KLHL3""","""KLHL3_Ala77Glu""","""ENSG00000146021""","""NC_000005.10:137709761:G:T""",1256,214008,"""230C>A""","""Ala77Glu""","""1_Pathogenic""",null,null,"""0.994""","""0.95096""","""P""","""-16.503""","""0.98425""","""D""",92.22,0.6037736,"""10-90""",""">90""",4.245714,"""Y""","""Increase""",0.741262,false,null,null
"""KLHL3""","""KLHL3_Glu85Ala""","""ENSG00000146021""","""NC_000005.10:137698396:T:G""",1256,214009,"""254A>C""","""Glu85Ala""","""1_Pathogenic""",null,null,"""0.9757""","""0.90318""","""P""","""-15.134""","""0.96183""","""D""",77.43,0.386598,"""10-90""","""70-90""",3.952288,"""Y""","""Increase""",0.786995,false,null,null
"""KLHL3""","""KLHL3_Cys164Phe""","""ENSG00000146021""","""NC_000005.10:137692320:C:A""",1256,214010,"""491G>T""","""Cys164Phe""","""1_Pathogenic""",null,null,"""0.9776""","""0.90676""","""P""","""-15.292""","""0.96508""","""D""",90.95,0.022222,"""<10""",""">90""",4.530291,"""Y""","""Increase""",0.851896,false,null,null
"""NAA10""","""NAA10_Arg116Trp""","""ENSG00000102030""","""NC_000023.11:153932111:G:A""",3180,214556,"""346C>T""","""Arg116Trp""","""1_Pathogenic""",null,2.0,"""0.9744""","""0.90095""","""P""","""-13.027""","""0.89498""","""D""",95.82,0.858871,"""10-90""",""">90""",2.654036,"""Y""","""Increase""",null,null,0.227046,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""MID1""","""MID1_Lys370Glu""","""ENSG00000101871""","""NC_000023.11:10474656:T:C""",11907,20187,"""1108A>G""","""Lys370Glu""","""6_No_ClinVar""",null,null,null,null,null,null,null,null,null,null,"""10-90""",""">90""",6.174735,"""Y""","""Increase""",null,null,null,null
"""ORC6""","""ORC6_Tyr232Ser""","""ENSG00000091651""","""NC_000016.10:46697521:A:C""",12043,203658,"""695A>C""","""Tyr232Ser""","""1_Pathogenic""",null,1.0,"""0.7252""","""0.74293""","""P""","""-11.731""","""0.83565""","""D""",81.28,0.5855856,"""10-90""","""70-90""",4.35952,"""Y""","""Increase""",null,null,null,null
"""PKP2""","""PKP2_Thr263Met""","""ENSG00000057294""","""NC_000012.12:32878092:G:A""",54308,3102,"""788C>T""","""Thr263Met""","""4_VUS""",0.000027,2.0,"""0.1218""","""0.25463""","""B""","""-5.02""","""0.37444""","""T""",29.71,0.866197,"""10-90""","""<=50""",2.55075,"""Y""","""Increase""",null,null,null,null


In [11]:
al_dual_img_ppi_key_cols.drop_nulls(subset=dualipa_cols+img_cols+ppi_cols)

symbol,gene_variant,ensembl_gene_id,spdi,orf_id,mut_id,nt_change,aa_change,clinvar_clnsig_clean,gnomad_af,StarStatus,AlphaMissense_score,AlphaMissense_rankscore,AlphaMissense_pred,ESM1b_score,ESM1b_rankscore,ESM1b_pred,plddt_f32,rsa_f32,RSA_Category,pLDDT_Category,dualipa_abun_score,dualipa_abun_change,dualipa_abun_chg_dir,AUROC_Mean_GFP,Altered_95th_perc_both_batches_GFP,ppi_score_final_euclidean,ppi_hit
str,str,str,str,i64,i64,str,str,str,f64,f64,str,str,str,str,str,str,f64,f64,str,str,f64,str,str,f64,bool,f64,bool
"""MVK""","""MVK_His20Pro""","""ENSG00000110921""","""NC_000012.12:109574881:A:C""",281,3,"""59A>C""","""His20Pro""","""1_Pathogenic""",0.00001,2.0,"""0.906""","""0.85569""","""LP""","""-14.191""","""0.93673""","""D""",94.91,0.043478,"""<10""",""">90""",1.094888,"""Uncertain""","""Increase""",0.662373,false,0.156925,false
"""MVK""","""MVK_Cys101Tyr""","""ENSG00000110921""","""NC_000012.12:109579877:G:A""",281,7,"""302G>A""","""Cys101Tyr""","""4_VUS""",null,1.0,"""0.0905""","""0.22167""","""LB""","""-4.048""","""0.24542""","""T""",84.84,0.074074,"""<10""","""70-90""",0.294497,"""N""","""Increase""",0.586852,false,0.138978,false
"""MVK""","""MVK_Pro167Leu""","""ENSG00000110921""","""NC_000012.12:109581523:C:T""",281,13,"""500C>T""","""Pro167Leu""","""1_Pathogenic""",0.000009,2.0,"""0.2005""","""0.42461""","""LB""","""-5.542""","""0.42311""","""T""",93.32,0.132353,"""10-90""",""">90""",0.836342,"""N""","""Increase""",0.673916,false,0.103865,false
"""MVK""","""MVK_Leu255Pro""","""ENSG00000110921""","""NC_000012.12:109590857:T:C""",281,18,"""764T>C""","""Leu255Pro""","""1_Pathogenic""",null,null,""".""","""0.62755""",""".""","""-10.926""","""0.79318""","""D""",98.07,0.75,"""10-90""",""">90""",-6.6344,"""Y""","""Decrease""",0.965357,true,0.082272,false
"""MVK""","""MVK_Ile268Val""","""ENSG00000110921""","""NC_000012.12:109591274:A:G""",281,19,"""802A>G""","""Ile268Val""","""3_Conflicting""",0.000005,1.0,""".""","""0.12948""",""".""","""-3.517""","""0.16684""","""T""",98.47,0.005917,"""<10""",""">90""",-0.558686,"""Uncertain""","""Increase""",0.684605,false,0.052964,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""KCNJ2""","""KCNJ2_Gly300Asp""","""ENSG00000123700""","""NC_000017.11:70175938:G:A""",100016069,2641,"""899G>A""","""Gly300Asp""","""1_Pathogenic""",null,2.0,"""0.9993""","""0.98903""","""P""","""-16.573""","""0.98503""","""D""",94.45,0.059524,"""<10""",""">90""",-4.97117,"""Y""","""Decrease""",0.989765,true,0.297519,false
"""KCNJ2""","""KCNJ2_Met301Leu""","""ENSG00000123700""","""NC_000017.11:70175940:A:C""",100016069,2643,"""901A>C""","""Met301Leu""","""3_Conflicting""",null,1.0,"""0.7319""","""0.74234""","""LP""","""-7.086""","""0.54729""","""T""",94.04,0.218085,"""10-90""",""">90""",-1.929122,"""Uncertain""","""Increase""",0.989451,true,0.325572,false
"""KCNJ2""","""KCNJ2_Met301Val""","""ENSG00000123700""","""NC_000017.11:70175940:A:G""",100016069,2644,"""901A>G""","""Met301Val""","""3_Conflicting""",6.8405e-7,1.0,"""0.7434""","""0.74721""","""P""","""-3.586""","""0.17668""","""T""",94.04,0.218085,"""10-90""",""">90""",-5.541692,"""Y""","""Decrease""",0.991546,true,0.088808,false


### 2.2 Gene Alleles Profiled by AT LEAST ONE assay

In [13]:
al_dual_img_ppi_key_cols.filter(
    ~pl.all_horizontal([pl.col(col).is_null() for col in dualipa_cols+img_cols+ppi_cols])
).unique("gene_variant").write_csv(
    "../3_outputs/all_gene_alleles_profiled_by_any_assay.csv"
)

### 2.3 Plot hits

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

al_dual_img_ppi_intersect = al_dual_img_ppi.filter(
    pl.col("dualipa_abun_score").is_not_null(),
    pl.col("AUROC_Mean_GFP").is_not_null(),
    pl.col("normalized_euclid_distance").is_not_null()
).unique("gene_variant")

df = al_dual_img_ppi_intersect.select(
    pl.col(["symbol", "gene_variant", "dualipa_abun_change", "Altered_95th_perc_both_batches_GFP", "hit", "clinvar_clnsig_clean"])
).with_columns(
    pl.when(pl.col("dualipa_abun_change")==1)
    .then(pl.lit(1))
    .otherwise(pl.lit(0))
    .alias("DUAL-IPA"),
    pl.col("Altered_95th_perc_both_batches_GFP").cast(pl.Int8).alias("Imaging"),
    pl.col("hit").alias("sqY2H"),
).to_pandas()

# Ensure hit columns are ints
hit_cols = ['DUAL-IPA', 'Imaging', 'sqY2H']
df[hit_cols] = df[hit_cols].astype(int)

# Sort by gene symbol and ClinVar category
df_sorted = df.sort_values(['symbol', 'clinvar_clnsig_clean'])

# Create the hit matrix
matrix = df_sorted.set_index('gene_variant')[hit_cols]

# Row colors for ClinVar categories
lut = palette_dict["clinvar_clnsig_clean"]
row_colors_clinvar = df_sorted['clinvar_clnsig_clean'].map(lut).values

# Split genes into three groups while keeping alleles together
unique_genes = df_sorted['symbol'].unique()
n_genes = len(unique_genes)
split_point1 = n_genes // 3
split_point2 = 2 * n_genes // 3

genes_part1 = unique_genes[:split_point1]
genes_part2 = unique_genes[split_point1:split_point2]
genes_part3 = unique_genes[split_point2:]

# Create three separate datasets
df_part1 = df_sorted[df_sorted['symbol'].isin(genes_part1)]
df_part2 = df_sorted[df_sorted['symbol'].isin(genes_part2)]
df_part3 = df_sorted[df_sorted['symbol'].isin(genes_part3)]

def create_clustermap_part(df_part, part_name, genes_subset):
    # Create matrix for this part
    matrix_part = df_part.set_index('gene_variant')[hit_cols]
    
    # Row colors for ClinVar categories
    row_colors_clinvar_part = df_part['clinvar_clnsig_clean'].map(lut).values
    
    # Row colors for gene symbols (use consistent colors)
    gene_palette = sns.color_palette("tab20", 20) + sns.color_palette("Set1", n_genes-20)
    gene_lut = dict(zip(unique_genes, gene_palette))  # Keep all genes for consistent colors
    row_colors_genes_part = [gene_lut[gene] for gene in df_part['symbol']]
    
    # Combine row colors
    row_colors_part = [row_colors_clinvar_part, row_colors_genes_part]
    
    # Create clustermap
    g = sns.clustermap(
        matrix_part,
        row_cluster=False,
        col_cluster=False,
        cmap=cmap,
        linewidths=0.5,
        row_colors=row_colors_part,
        figsize=(5,12),
        colors_ratio=0.1,  # Increase from default 0.03 to make row colors wider
        cbar_pos=(-0.2, 0.45, 0.05, 0.18),
        cbar_kws={
            "ticks": [0, 1],
            "format": "%d",
            "format": lambda x, pos: "False" if x == 0 else "True",
        },
        dendrogram_ratio=0.01,  # Remove dendrogram space
        yticklabels=1
    )
    
    # Add legend for ClinVar categories
    handles = [Patch(facecolor=color, label=label) for label, color in lut.items()]
    g.ax_heatmap.legend(
        handles=handles,
        title='ClinVar',
        bbox_to_anchor=(-1., 0.35),
        loc='center left',
        frameon=False
    )
    g.ax_heatmap.tick_params(axis='x', rotation=45)
    # g.ax_heatmap.set_yticks(range(len(matrix_part)))
    # g.ax_heatmap.set_yticklabels(matrix_part.index, fontsize=10, va='center')

    # Change colorbar labels
    # g.ax_cbar.set_yticklabels(['False', 'True'])
    
    # plt.suptitle(f'{part_name} (Genes: {", ".join(genes_subset[:5])}{"..." if len(genes_subset) > 5 else ""})', 
    #              y=1., fontsize=14)
    plt.show()

# Create both parts
print(f"Part 1: {len(genes_part1)} genes, {len(df_part1)} variants")
create_clustermap_part(df_part1, "Part 1", genes_part1)

print(f"Part 2: {len(genes_part2)} genes, {len(df_part2)} variants")
create_clustermap_part(df_part2, "Part 2", genes_part2)

print(f"Part 2: {len(genes_part2)} genes, {len(df_part2)} variants")
create_clustermap_part(df_part3, "Part 3", genes_part3)